# Merged receiver and fair handoff timing (twelve carriers), Kaggle T4

No training. The frozen epoch-1 query adapter (sha256 864a8f19...) is folded into B's q_proj weights, computed in fp32 and rounded once to fp16. The merged weights are kept beside the originals and swapped in only for B, so A, support construction and every baseline still run on the exact original model.

Run in order:
1. cells 1-3: setup, pinned packages, upstream code and the unit tests;
2. cell 4 (`agree`, ~20 min): merged vs unmerged on the 24 dev sheets. Forced logits, labels, final choices and formatting are compared against registered tolerances. **Stop here if it does not pass**;
3. cell 5 (`integrate`, ~25 min): on the same dev sheets, the frozen R vs the R that A's GPU selection and re-rotation actually produce, each passed through the merged receiver. It also runs the direct path the benchmark times (R kept on the GPU, shared-prefix generation). **If it does not pass, stop and send the output: the difference must be documented before any deployment timing is interpreted**;
4. cell 6 (`bench --deployment same_gpu --limit 1`): one eval sheet, to check the handoff and timing rows;
5. cell 7 (`bench --deployment same_gpu`, ~3 h, resumable): six receivers on all 48 eval sheets, with warm-up, three repetitions and a random condition order;
6. cell 8 (`bench --deployment device_transfer`, ~3 h, optional): the same measurement with device-to-host and host-to-device copies;
7. cell 9 (`textadapt`, ~10 min): diagnostic only.

Deployments being measured:
- `same_gpu`: direct reuse on one GPU. A's result states are selected and re-rotated on the GPU and placed straight into B's cache; nothing leaves the device.
- `device_transfer`: a transfer between devices, approximated on one T4 by copying what B receives device-to-host and back. The network between hosts is **not** measured; bytes are reported.

An earlier version of `bench` (files `bench_evaluations.jsonl` and `bench_summary.json`, if you ran it) combined GPU-side preparation with a host-to-GPU copy of the frozen R. It matches neither deployment and is not used for deployment conclusions.

The eval sheets are reused as timing inputs. This is an implementation and timing comparison, not another independent confirmation.

Upload the repo including the `results_stage_plan4_qadapt/` folder (`frozen_candidate.json`, `adapter_epoch1.pt`, `dev_evaluations.jsonl`, `eval_evaluations.jsonl`, `a_artifacts_eval.jsonl`, `train_summary.json`, `prep_*`). Results go to `/kaggle/working/results_stage_plan4_qmerge`; the qadapt folder is only read.


In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
STAGE = "stage_plan4_qmerge"

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if f"{STAGE}.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / f"{STAGE}.py").is_file() and (cand / "data" / "plan4_qadapt_eval.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError(f"Need src/{STAGE}.py and data/plan4_qadapt_eval.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / f"{STAGE}.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
qa = [d for d in (REPO / "results_stage_plan4_qadapt", WORK / "results_stage_plan4_qadapt") if d.is_dir()]
for name in ("frozen_candidate.json", "adapter_epoch1.pt", "dev_evaluations.jsonl", "eval_evaluations.jsonl",
             "a_artifacts_eval.jsonl", "train_summary.json", "prep_a_artifacts_dev.jsonl",
             "prep_timing12_a_artifacts_12.jsonl", "prep_check_summary.json"):
    print("qadapt", name, any((d / name).is_file() for d in qa))
for name in ("plan4_qadapt_dev.jsonl", "plan4_qadapt_eval.jsonl", "plan4_timing12.jsonl"):
    print("data", name, (REPO / "data" / name).is_file())

def out_file(name):
    for d in (WORK / "results_stage_plan4_qmerge", REPO / "results" / STAGE / "run"):
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(name)

def jsonl(name):
    return [json.loads(l) for l in out_file(name).read_text(encoding="utf-8").splitlines() if l.strip()]


In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu, "| torch", torch.__version__)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest"])
rc = subprocess.call([sys.executable, "-m", "pytest", "-q", "tests/test_qmerge.py"])
print("unit tests exit", rc)
if rc != 0:
    raise RuntimeError("Unit tests failed; do not run the stage.")
print("Dest /kaggle/working/results_stage_plan4_qmerge. Implementation and timing comparison on reused inputs.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "agree"])
print("agree exit", rc)
a = json.loads(out_file("agree_summary.json").read_text(encoding="utf-8"))
print("merge s", round(a["merge"]["merge_s"], 3), "| worst rounding", a["merge"]["stats"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "answer_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    for mode in ("unmerged", "merged"):
        print(v, mode, {k: a["labels"][v][mode].get(k) for k in keys}, "| choice", a["final_choice"][v][mode])
print("\nforced logits", a["logits"])
print("label predictions changed", a.get("label_predictions_changed"))
print("unmerged rerun vs saved dev rows", a["unmerged_rerun_matches_saved"])
for name, g in a["gates"].items():
    print("gate", name, g)
print("\ndifferences")
for d in a["differences"]:
    print(" ", d)
print("\nPASSED" if a["passed"] else "\nNOT PASSED: do not run the benchmark; send these results.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "integrate"])
print("integrate exit", rc)
i = json.loads(out_file("integrate_summary.json").read_text(encoding="utf-8"))
for name, desc in i["runs"].items():
    print(name, "=", desc)
print("\nR, frozen vs GPU-prepared", i["r_comparison"])
print("forced logits, frozen R vs GPU R (merged receiver)", i["logits_frozen_vs_actual_r"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "answer_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    for run in i["runs"]:
        print(v, run, {k: i["labels"][v][run].get(k) for k in keys}, "| choice", i["final_choice"][v][run]["final_choice_correct"])
print("\nfrozen_std reproduces the agree merged rows", i["frozen_std_reproduces_agree_merged"])
for block in ("actual_r_vs_frozen_r", "direct_path_vs_frozen_r"):
    print("\n" + block, "passed", i[block]["passed"], "| label predictions changed", i[block]["label_predictions_changed"])
    for name, g in i[block]["gates"].items():
        print("  gate", name, g)
print("\ndifferences")
for run, diffs in i["differences"].items():
    for d in diffs:
        print(" ", run, d)
print("\nPASSED" if i["passed"] else f"\nNOT PASSED ({i['cause']}): do not run the benchmark; send these results.")
print(i["next"])

In [ ]:
DEPLOYMENT = "same_gpu"
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "bench", "--deployment", DEPLOYMENT, "--limit", "1"])
print("bench (first sheet) exit", rc)
c = jsonl(f"bench_{DEPLOYMENT}_checks.jsonl")[0]
print("checks", {k: v for k, v in c.items() if k != "bytes"})
print("bytes", c["bytes"])
parts = ("sender_prep_s", "movement_s", "movement_out_s", "movement_in_s", "receiver_prep_s", "r_prefill_s", "encode_s",
         "shared_prefill_s", "plan_prefill_s", "plan_decode_s", "answer_prefill_s", "answer_decode_s", "crop_s", "untimed_bench_copy_s")
for r in jsonl(f"bench_{DEPLOYMENT}_evaluations.jsonl"):
    print(r["rep"], r["order_index"], r["condition"], "| labels", r["plan_n_status_correct"], "| choice", r["correct"],
          "| total", round(r["handoff_total_s"], 3), "| sum", round(r["component_sum_s"], 3),
          "| shared/plan/answer tokens", r["n_shared_tokens"], r["n_plan_suffix_tokens"], r["n_answer_suffix_tokens"],
          "| gen", r["n_plan_tokens"], r["n_new_tokens"])
    print("    ", {p: round(r[p], 4) for p in parts})

In [ ]:
DEPLOYMENT = "same_gpu"
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "bench", "--deployment", DEPLOYMENT])
print("bench exit", rc)
s = json.loads(out_file(f"bench_{DEPLOYMENT}_summary.json").read_text(encoding="utf-8"))
print("deployment", s["deployment"])
print("integration", s["integration"])
print("complete", s["complete"], "| rows", s["n_rows"], "| checks ok", s["checks_ok"], "| worst", s["checks_worst"])
print("original weights", s["original_weights"], "| merge", s["merge"])
print("one-time", s["one_time"])
print("public prefixes", s["public_prefixes"])
keys = ("labels_correct", "cases_all_twelve_correct", "parse_ok", "plan_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    print(f"\n{v} (repetition 1)")
    for cond, b in s["labels"][v].items():
        print(" ", cond, {k: b.get(k) for k in keys}, "| choice", s["final_choice"][v][cond]["final_choice_correct"])
print("\nconsistency across reps", s["consistency_across_reps"])
print("vs saved eval rows", s["vs_saved_eval_rows"])
print("\ntiming (mean over sheets of the per-sheet median, seconds)")
cols = ("preparation_total_s", "sender_prep_s", "movement_s", "movement_out_s", "movement_in_s", "receiver_prep_s", "r_prefill_s", "encode_s", "untimed_bench_copy_s",
        "prefill_total_s", "decode_total_s", "decode_s_per_token", "crop_s", "handoff_total_s", "component_sum_s",
        "n_kv_received", "bytes_moved", "peak_gpu_gb")
for cond, t in s["timing"].items():
    print(" ", cond, {k: (round(t[k], 4) if isinstance(t[k], float) else t[k]) for k in cols})
    print("    total by rep", {k: round(v, 3) for k, v in t["handoff_total_by_rep"].items()})
print("\nmerged minus comparison, complete handoff")
for other, p in s["paired_handoff_time"].items():
    print(" ", other, {k: p.get(k) for k in ("mean_delta_per_case", "ci95_per_case", "mean_delta_by_rep", "repeatable_saving", "repeatable_cost")})
print("\nmerged minus comparison, by component")
for other, comp in s["paired_components"].items():
    print(" ", other, comp)
print("\nreliability", s["reliability_check"])
print("\nDECISION", json.dumps(s["decision"], indent=1))

In [ ]:
DEPLOYMENT = "device_transfer"  # optional; network between hosts not measured
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "bench", "--deployment", DEPLOYMENT])
print("bench exit", rc)
s = json.loads(out_file(f"bench_{DEPLOYMENT}_summary.json").read_text(encoding="utf-8"))
print("deployment", s["deployment"])
print("complete", s["complete"], "| rows", s["n_rows"], "| checks ok", s["checks_ok"])
keys = ("labels_correct", "cases_all_twelve_correct", "parse_ok", "plan_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    print(f"\n{v} (repetition 1)")
    for cond, b in s["labels"][v].items():
        print(" ", cond, {k: b.get(k) for k in keys}, "| choice", s["final_choice"][v][cond]["final_choice_correct"])
print("\ntiming (mean over sheets of the per-sheet median, seconds)")
cols = ("preparation_total_s", "sender_prep_s", "movement_s", "movement_out_s", "movement_in_s", "receiver_prep_s", "r_prefill_s",
        "prefill_total_s", "decode_total_s", "handoff_total_s", "bytes_moved")
for cond, t in s["timing"].items():
    print(" ", cond, {k: (round(t[k], 4) if isinstance(t[k], float) else t[k]) for k in cols})
print("\nmerged minus comparison, complete handoff")
for other, p in s["paired_handoff_time"].items():
    print(" ", other, {k: p.get(k) for k in ("mean_delta_per_case", "ci95_per_case", "mean_delta_by_rep", "repeatable_saving", "repeatable_cost")})
print("\nreliability", s["reliability_check"])
print("\nDECISION", json.dumps(s["decision"], indent=1))

In [ ]:
rc = subprocess.call([sys.executable, "-m", f"src.{STAGE}", "--phase", "textadapt"])
print("textadapt exit", rc)
t = json.loads(out_file("textadapt_summary.json").read_text(encoding="utf-8"))
print(t["scope"])
keys = ("labels_correct", "cases_all_twelve_correct", "parse_ok", "plan_truncated", "plan_lines_missing")
for v in ("base", "a_flip"):
    for cond, b in t["labels"][v].items():
        print(v, cond, {k: b.get(k) for k in keys})
    for cond, b in t["final_choice"][v].items():
        print(v, cond, "choice", b)
print("complete", t["complete"], "| original weights", t["original_weights"])